In [273]:
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import OneHotEncoder
from sklearn.compose import ColumnTransformer, make_column_selector
from sklearn.pipeline import Pipeline
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import classification_report, confusion_matrix
from sklearn.impute import SimpleImputer


In [274]:
data = pd.read_csv("TOTAL_KSI_3737821728629277523.csv")
print(data.shape)
data = data.dropna(subset=['ACCLASS'])
data = data[data["ACCLASS"] != "Property Damage O"]
print(data.shape)
print(data["ACCLASS"].value_counts())


(18957, 54)
(18938, 54)
Non-Fatal Injury    16268
Fatal                2670
Name: ACCLASS, dtype: int64


In [275]:
data_new = data.drop(['OBJECTID','INDEX', 'ACCNUM', 'STREET1', 'STREET2', 'OFFSET', 'LATITUDE', 
                  'LONGITUDE', 'INJURY', 'FATAL_NO', 'VEHTYPE', 'HOOD_158', 'HOOD_140', 
                  'NEIGHBOURHOOD_140', 'DIVISION', 'x', 'y', 'NEIGHBOURHOOD_158'], axis=1)

print(data_new.columns)

Index(['DATE', 'TIME', 'ROAD_CLASS', 'DISTRICT', 'ACCLOC', 'TRAFFCTL',
       'VISIBILITY', 'LIGHT', 'RDSFCOND', 'ACCLASS', 'IMPACTYPE', 'INVTYPE',
       'INVAGE', 'INITDIR', 'MANOEUVER', 'DRIVACT', 'DRIVCOND', 'PEDTYPE',
       'PEDACT', 'PEDCOND', 'CYCLISTYPE', 'CYCACT', 'CYCCOND', 'PEDESTRIAN',
       'CYCLIST', 'AUTOMOBILE', 'MOTORCYCLE', 'TRUCK', 'TRSN_CITY_VEH',
       'EMERG_VEH', 'PASSENGER', 'SPEEDING', 'AG_DRIV', 'REDLIGHT', 'ALCOHOL',
       'DISABILITY'],
      dtype='object')


In [276]:
X = data_new.drop("ACCLASS", axis=1)
y = data_new['ACCLASS']

In [277]:
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)
print(f"Train set size: {X_train.shape}")
print(f"Test set size: {X_test.shape}")

Train set size: (15150, 35)
Test set size: (3788, 35)


In [278]:
cols_with_nan = X_train.columns[X_train.isnull().any()].tolist()
nan_types = X_train[cols_with_nan].dtypes
print(cols_with_nan)

['ROAD_CLASS', 'DISTRICT', 'ACCLOC', 'TRAFFCTL', 'VISIBILITY', 'LIGHT', 'RDSFCOND', 'IMPACTYPE', 'INVTYPE', 'INITDIR', 'MANOEUVER', 'DRIVACT', 'DRIVCOND', 'PEDTYPE', 'PEDACT', 'PEDCOND', 'CYCLISTYPE', 'CYCACT', 'CYCCOND', 'PEDESTRIAN', 'CYCLIST', 'AUTOMOBILE', 'MOTORCYCLE', 'TRUCK', 'TRSN_CITY_VEH', 'EMERG_VEH', 'PASSENGER', 'SPEEDING', 'AG_DRIV', 'REDLIGHT', 'ALCOHOL', 'DISABILITY']


In [279]:
fill_values = {
    col: 'No' if 'Yes' in X_train[col].unique() else 'Unknown' 
    for col in cols_with_nan
}

X_train = X_train.fillna(value=fill_values)
X_test = X_test.fillna(value=fill_values)

print(f"NaN in Train: {X_train.isnull().sum().sum()}")
print(f"NaN in Test: {X_test.isnull().sum().sum()}")

NaN in Train: 0
NaN in Test: 0


In [280]:
def build_features(df):
    df = df.copy()
    # change to pd date format
    df['DATE'] = pd.to_datetime(df['DATE'])
    df['HOUR'] = df['TIME'] // 100
    season_map = {
        12: 'Winter', 1: 'Winter', 2: 'Winter',
        3: 'Spring', 4: 'Spring', 5: 'Spring',
        6: 'Summer', 7: 'Summer', 8: 'Summer',
        9: 'Autumn', 10: 'Autumn', 11: 'Autumn'
    }
    df['SEASON'] = df['DATE'].dt.month.map(season_map)

    df['IS_WEEKEND'] = df['DATE'].dt.dayofweek.apply(lambda x: 1 if x >= 5 else 0)

    # Time_Range
    # -1 cause 00:00 will be included only in this way, pandas read (0, 5] NOT [0, 5]
    bins = [-1, 5, 9, 14, 19, 24]
    labels = ['Night', 'Morning_Rush', 'Day', 'Afternoon_Rush', 'Evening']

    df['TIME_RANGE'] = pd.cut(df['HOUR'], bins=bins, labels=labels)

    age_map = {
        '0 to 4': 'Child', '5 to 9': 'Child', '10 to 14': 'Child',
        '15 to 19': 'Young_Adult', '20 to 24': 'Young_Adult',
        '25 to 29': 'Adult', '30 to 34': 'Adult', '35 to 39': 'Adult',
        '40 to 44': 'Adult', '45 to 49': 'Adult', '50 to 54': 'Adult',
        '55 to 59': 'Adult', '60 to 64': 'Adult',
        '65 to 69': 'Senior', '70 to 74': 'Senior', '75 to 79': 'Senior',
        '80 to 84': 'Senior', '85 to 89': 'Senior', '90 to 94': 'Senior',
        'Over 95': 'Senior',
        'unknown': 'Unknown', 'Unknown': 'Unknown'
    }
    df['AGE_GROUP'] = df['INVAGE'].map(age_map)

    # if NaN after map() - change to Unknown
    df['AGE_GROUP'] = df['AGE_GROUP'].fillna('Unknown')

    # Convert Yes/nan columns to binary 1/0
    yes_no_cols = [
        "PEDESTRIAN","CYCLIST","AUTOMOBILE","MOTORCYCLE","TRUCK",
        "TRSN_CITY_VEH","EMERG_VEH","PASSENGER",
        "SPEEDING","AG_DRIV","REDLIGHT","ALCOHOL","DISABILITY"
    ]

    for col in yes_no_cols:
        if col in df.columns:
            df[col] = df[col].apply(lambda x: 1 if str(x) == "Yes" else 0)

    # DRIVER IMPAIRMENT (based on DRIVCOND)
    driver_impaired_values = [
        "Ability Impaired, Alcohol Over .08",
        "Ability Impaired, Alcohol",
        "Ability Impaired, Drugs",
        "Had Been Drinking",
        "Medical or Physical Disability"
    ]

    df["DRIVER_IMPAIRED"] = df["DRIVCOND"].isin(driver_impaired_values).astype(int)

    # DRIVER INATTENTION (based on DRIVCOND)
    driver_inattention_values = [
        "Inattentive",
        "Fatigue"
    ]

    df["DRIVER_INATTENTION"] = df["DRIVCOND"].isin(driver_inattention_values).astype(int)

    # DRIVER RISK SCORE (binary flags + impairment)
    risk_cols = ["SPEEDING","AG_DRIV","REDLIGHT","ALCOHOL","DISABILITY","DRIVER_IMPAIRED"]
    risk_cols = [c for c in risk_cols if c in df.columns]

    df["DRIVER_RISK_SCORE"] = df[risk_cols].sum(axis=1)
    df["HIGH_RISK_DRIVER"] = (df["DRIVER_RISK_SCORE"] >= 2).astype(int)

    # AGGRESSIVE MANEUVER (MANOEUVER + DRIVACT)
    aggressive_manoeuvre_values = [
        "Changing Lanes",
        "Turning Right",
        "Turning Left",
        "Overtaking",
        "Making U Turn",
        "Merging"
    ]

    aggressive_drivact_values = [
        "Exceeding Speed Limit",
        "Speed too Fast For Condition",
        "Improper Passing",
        "Improper Lane Change",
        "Following too Close",
        "Improper Turn",
        "Lost control"
    ]

    df["AGGRESSIVE_MANEUVER"] = (
        df["MANOEUVER"].isin(aggressive_manoeuvre_values) &
        df["DRIVACT"].isin(aggressive_drivact_values)
    ).astype(int)

    # VULNERABLE ROAD USERS
    df["VULNERABLE_USER"] = ((df["PEDESTRIAN"] == 1) | (df["CYCLIST"] == 1)).astype(int)
    df["NUM_VULNERABLE_USERS"] = df["PEDESTRIAN"] + df["CYCLIST"]

    # VEHICLE COUNT
    vehicle_cols = ["AUTOMOBILE","MOTORCYCLE","TRUCK","TRSN_CITY_VEH","EMERG_VEH"]
    vehicle_cols = [c for c in vehicle_cols if c in df.columns]

    df["NUM_VEHICLES_INVOLVED"] = df[vehicle_cols].sum(axis=1)

    # VEHICLE CLASS (based on VEHTYPE values you provided)
    vehicle_map = {
        "Going Ahead": "Moving",
        "Changing Lanes": "Moving",
        "Turning Right": "Turning",
        "Turning Left": "Turning",
        "Making U Turn": "Turning",
        "Merging": "Merging",
        "Overtaking": "Overtaking",
        "Slowing or Stopping": "Slowing",
        "Stopped": "Stopped",
        "Reversing": "Reversing",
        "Parked": "Parked",
        "Pulling Away from Shoulder or Curb": "Pulling Away",
        "Pulling Onto Shoulder or towardCurb": "Pulling Toward",
        "Disabled": "Disabled",
        "Other": "Other",
        "Unknown": "Unknown",
        np.nan: "Unknown"
    }

    df["MANOEUVER_CLASS"] = df["MANOEUVER"].map(vehicle_map)

    # PEDESTRIAN RISK BEHAVIOR
    ped_risk_values = [
        "Crossing without right of way",
        "Crossing with right of way",
        "Crossing, no Traffic Control",
        "Crossing, Pedestrian Crossover",
        "Crossing marked crosswalk without ROW",
        "Running onto Roadway",
        "Coming From Behind Parked Vehicle",
        "Walking on Roadway Against Traffic",
        "Walking on Roadway with Traffic"
    ]

    df["PEDESTRIAN_RISK"] = df["PEDACT"].isin(ped_risk_values).astype(int)

    # CYCLIST RISK BEHAVIOR
    cyc_risk_values = [
        "Improper Turn",
        "Improper Passing",
        "Improper Lane Change",
        "Following too Close",
        "Lost control",
        "Speed too Fast For Condition",
        "Wrong Way on One Way Road",
        "Failed to Yield Right of Way"
    ]

    df["CYCLIST_RISK"] = df["CYCACT"].isin(cyc_risk_values).astype(int)

    # INTERACTION FEATURES
    df["RISKY_DRIVER_WITH_VULNERABLE_USER"] = (
        (df["HIGH_RISK_DRIVER"] == 1) & (df["VULNERABLE_USER"] == 1)
    ).astype(int)

    df["MULTI_VEHICLE_PEDESTRIAN"] = (
        (df["NUM_VEHICLES_INVOLVED"] > 1) & (df["PEDESTRIAN"] == 1)
    ).astype(int)

    df["IMPAIRED_AND_AGGRESSIVE"] = (
        (df["DRIVER_IMPAIRED"] == 1) & (df["AGGRESSIVE_MANEUVER"] == 1)
    ).astype(int)
    
    return df

In [281]:
X_train = build_features(X_train)
X_test = build_features(X_test)

print('Train:', X_train_encoded.shape)
print('Test:', X_test_encoded.shape)

Train: (15164, 235)
Test: (3792, 235)


In [ ]:
# Creating preprocessing pipeline using ColumnTransformer
# For categorical columns: impute missing values (most frequent) and apply one-hot encoding
# For numeric columns: impute missing values using median
# Drop all other columns to ensure clean input for the model
categorical_cols = X_train.select_dtypes(include=["object", "category"]).columns.tolist()

preprocess = ColumnTransformer(
    transformers=[
        ("cat", Pipeline([
            ("impute", SimpleImputer(strategy="most_frequent")),
            ("onehot", OneHotEncoder(handle_unknown="ignore"))
        ]), categorical_cols),

        ("num", Pipeline([
            ("impute", SimpleImputer(strategy="median"))
        ]), make_column_selector(dtype_include=np.number))
    ],
    remainder="drop"
)


In [283]:
model = Pipeline(steps=[
    ("preprocess", preprocess),
    ("clf", RandomForestClassifier(
        n_estimators=300,
        max_depth=None,
        random_state=42,
        class_weight="balanced"
    ))
])


In [284]:
model.fit(X_train, y_train)

ValueError: A given column is not a column of the dataframe

In [ ]:
# y_pred = model.predict(X_test)

# print("\nClassification Report:")
# print(classification_report(y_test, y_pred))

# print("\nConfusion Matrix:")
# print(confusion_matrix(y_test, y_pred))

In [ ]:
# trying to improve with threshold tuning
y_proba = model.predict_proba(X_test)
fatal_index = list(model.classes_).index("Fatal")

threshold = 0.2  # default is ~0.33

y_pred = np.where(
    y_proba[:, fatal_index] > threshold,
    "Fatal",
    "Non-Fatal Injury"
)

print("\nClassification Report:")
print(classification_report(y_test, y_pred))

print("\nConfusion Matrix:")
print(confusion_matrix(y_test, y_pred))

NotFittedError: This ColumnTransformer instance is not fitted yet. Call 'fit' with appropriate arguments before using this estimator.